## Demo 6: Event streaming

Demo 4 asked *how do you know what changed?* and answered it three times: compare `MAX(id)`,
compare `updated_at`, or let Change Data Capture write it down for you.

All three have the same shape. They stand at the target, look at the source, and work out the
difference. Nobody ever asked the application what it did.

This demo does. The PhotoService shop knows perfectly well when it added a customer - it is the
thing that added them - and it can say so, once, on a log that anybody may read at their own speed.

**This is the only part of this repository that is not a port.** The sibling repository has no
Kafka, so this is the one demo with nothing to stand beside it. Its own equivalent section is
called *"Transfer data from logging (or kafka)"*, though - so the replay loop below is still
recognisably a translation of it, and only the transport underneath is different.

> **Give the shop two minutes before running this.** `docker/photoservice-app.py` keeps the
> sibling's schedule: a customer every six seconds from the start, the first order after 60
> seconds, the first payment after 90, the first shipment after 120. It also truncates its tables when it
> starts. Run this notebook straight after `docker compose up -d` and the topic will hold nothing but
> `Added customer`, `order_event` will be empty, and every count below will be zero - not because
> anything is broken, but because nothing has happened yet.
>
> `docker compose logs -f photoservice` shows how far along it is.

### Before Kafka: you may already have an outbox

There is a table in this database that nothing in demos 1 to 5 has ever read.

`order_event` gets a row every time an order is paid for or shipped. The application writes it in
the same transaction as the change itself, which is what makes it trustworthy: if the order did not
happen, neither did the row.

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path("../lib").resolve()))

from connect_pg_instance import connect_pg_instance
from invoke_pg_query import invoke_pg_query

pg_connection = connect_pg_instance(
    instance="127.0.0.1",
    database="photoservice",
    username="photoservice",
    password="Passw0rd!"
)

invoke_pg_query(
    connection=pg_connection,
    query="SELECT * FROM order_event ORDER BY id DESC LIMIT 5"
)

[VERBOSE] Creating connection to instance [127.0.0.1]
[VERBOSE] Using password authentication
[VERBOSE] Opening connection
[VERBOSE] Returning connection object
[VERBOSE] Creating cursor
[VERBOSE] Executing query
[VERBOSE] Retrieved 5 rows


,id,order_id,updated_at,payment_uuid,shipment_uuid
0,8527,4322,2026-08-07 09:28:06.680,None,bb555ed6-b3d7-4cbc-914c-5013ef3b35dd
1,8526,4660,2026-08-07 09:28:06.667,1f153099-a890-4176-ae1a-c87c71363b27,None
2,8525,4155,2026-08-07 09:28:05.627,None,63ad5462-ad19-46a6-9ee6-8bedc82fb8f5
3,8524,3839,2026-08-07 09:28:05.619,68953294-328f-49bf-a28f-1018af8bd72b,None
4,8523,4527,2026-08-07 09:28:04.596,None,92a131d1-aaf5-4b93-9617-a373bd91d660


That is a perfectly good pattern and it needs no new infrastructure at all. It is also where its
limits are easiest to see:

* **You poll it.** The reader asks every few seconds whether anything is new. Nothing pushes.
* **Deleting is your problem.** Rows stay until somebody removes them, and removing them is only
  safe once *every* reader is past them - so the table has to know about its readers.
* **A second reader changes the design.** One consumer can delete as it goes. Two cannot, and now
  you need a per-reader position, which is a table of offsets you have to write yourself.

Kafka is what that turns into when you keep solving those three problems. The log keeps the
messages, each reader keeps its own position in it, and nobody deletes anything on anybody else's
behalf.

### The same events, on a topic

`docker/photoservice-app.py` produces an event whenever something actually happens - a customer, an
order header, its details, a payment, a shipment. The five events keep the shape the sibling's
`Add-LoggingEvent` gives them, which is why the replay further down is still recognisably its loop.

Two connect functions, because Kafka has no single connection: producing and consuming are
different clients. The notebook only consumes.

In [2]:
from datetime import datetime

from connect_kfk_consumer import connect_kfk_consumer
from read_kfk_topic import read_kfk_topic

TOPIC = "photoservice.events"

# A group that has read before carries on where it stopped, which is the right behaviour and the
# wrong one for a notebook that gets stepped through again and again. So each run gets its own
# groups and therefore its own clean start.
RUN = datetime.now().strftime("%H%M%S")

# 127.0.0.1:19092 from Windows; the application inside the compose network uses redpanda:9092.
# The broker advertises both - see docker-compose.yaml.
consumer = connect_kfk_consumer(
    instance="127.0.0.1:19092",
    group_id=f"look-{RUN}",
    from_beginning=True
)

[VERBOSE] Creating consumer for instance [127.0.0.1:19092] in group [look-112810]
[VERBOSE] Checking the connection
[VERBOSE] Returning consumer


In [3]:
events = read_kfk_topic(connection=consumer, topic=TOPIC, first=5, as_type="dict")

events[0]

[VERBOSE] Subscribing to topic photoservice.events
[VERBOSE] Reading, stopping after 5 messages
[VERBOSE] Retrieved 5 messages


{'Timestamp': '2026-08-07T07:51:17.673+00:00',
 'Hostname': '28be4291e685',
 'Appname': 'PhotoService',
 'Component': 'Customer',
 'Level': 'INFO',
 'Message': 'Added customer',
 'Details': {'id': 1,
  'firstname': 'James',
  'surname': 'Brooks',
  'city': 'Hannover',
  'email': 'James.Brooks@hannover.de'}}

A timestamp, where it came from, what happened, and the details of what happened. `Details` is the
object the application was working with at that moment - not a row of a table, and not a diff
against anything.

Note what the numbers did on the way. JSON has no date and no decimal, so a `datetime` and a
`Decimal` both arrive as strings. That is the same question MongoDB asked with `Decimal128` and the
same answer: the transport will not guess, so the sender decides. SQL Server converts them back on
the way in, which is why the replay below can bind them as they are.

In [4]:
import pandas as pd

pd.DataFrame(events)[["Timestamp", "Component", "Message"]]

,Timestamp,Component,Message
0,2026-08-07T07:51:17.673+00:00,Customer,Added customer
1,2026-08-07T07:52:17.993+00:00,Customer,Added customer
2,2026-08-07T07:53:18.102+00:00,Customer,Added customer
3,2026-08-07T07:54:18.119+00:00,Customer,Added customer
4,2026-08-07T07:55:18.430+00:00,Customer,Added customer


### Replaying the events into SQL Server

This is the sibling's loop. Every message says what happened, so the consumer only has to know what
each kind of event means: three of them are rows to insert, two are updates to an order that already
exists.

In [5]:
# pandas again, because everything below builds a DataFrame - the cell above that first imported
# it only displays a table, and a cell that only displays something is a cell somebody will skip
import pandas as pd

from connect_sql_instance import connect_sql_instance
from invoke_sql_query import invoke_sql_query
from write_sql_table import write_sql_table

sql_connection = connect_sql_instance(
    instance="127.0.0.1",
    database="PhotoService",
    username="PhotoService",
    password="Passw0rd!"
)

# Dropped first, because this notebook is built to be run again. A fresh kernel invents a new
# group id, so the consumer starts at the beginning of the topic - and replaying those events
# into tables that still hold them is a primary key violation, not a demo.
for query in [
    "DROP TABLE IF EXISTS dbo.customer",
    "DROP TABLE IF EXISTS dbo.order_header",
    "DROP TABLE IF EXISTS dbo.order_detail",
    "CREATE TABLE dbo.customer (id INT, firstname VARCHAR(50), surname VARCHAR(50), city VARCHAR(50), email VARCHAR(200), CONSTRAINT customer_pk PRIMARY KEY (id))",
    "CREATE TABLE dbo.order_header (id INT, customer_id INT, created_at DATETIME2, updated_at DATETIME2, payment_uuid UNIQUEIDENTIFIER, shipment_uuid UNIQUEIDENTIFIER, CONSTRAINT order_header_pk PRIMARY KEY (id))",
    "CREATE TABLE dbo.order_detail (order_id INT, photo_id INT, quantity INT, price NUMERIC(7, 2), CONSTRAINT order_detail_pk PRIMARY KEY (order_id, photo_id))"
]:
    invoke_sql_query(connection=sql_connection, query=query, enable_exception=True)

[VERBOSE] Creating connection to instance [127.0.0.1]
[VERBOSE] Using SQL authentication
[VERBOSE] Disabling connection pooling
[VERBOSE] Opening connection
[VERBOSE] Returning connection object
[VERBOSE] Creating cursor
[VERBOSE] Executing query
[VERBOSE] Non-query executed, rowcount=-1
[VERBOSE] Creating cursor
[VERBOSE] Executing query
[VERBOSE] Non-query executed, rowcount=-1
[VERBOSE] Creating cursor
[VERBOSE] Executing query
[VERBOSE] Non-query executed, rowcount=-1
[VERBOSE] Creating cursor
[VERBOSE] Executing query
[VERBOSE] Non-query executed, rowcount=-1
[VERBOSE] Creating cursor
[VERBOSE] Executing query
[VERBOSE] Non-query executed, rowcount=-1
[VERBOSE] Creating cursor
[VERBOSE] Executing query
[VERBOSE] Non-query executed, rowcount=-1


In [6]:
def apply_event(event):
    details = event["Details"]

    if event["Message"] == "Added customer":
        write_sql_table(connection=sql_connection, table="dbo.customer",
                        data=pd.DataFrame([details]), enable_exception=True)

    elif event["Message"] == "Added order header":
        write_sql_table(connection=sql_connection, table="dbo.order_header",
                        data=pd.DataFrame([details]), enable_exception=True)

    elif event["Message"] == "Added order details":
        # The only event whose Details is a list - one order, many lines
        write_sql_table(connection=sql_connection, table="dbo.order_detail",
                        data=pd.DataFrame(details), enable_exception=True)

    elif event["Message"] == "Added payment":
        invoke_sql_query(
            connection=sql_connection,
            query="UPDATE dbo.order_header SET updated_at = @updated_at, payment_uuid = @payment_uuid WHERE id = @order_id",
            parameter_values=details, enable_exception=True
        )

    elif event["Message"] == "Added shipment":
        invoke_sql_query(
            connection=sql_connection,
            query="UPDATE dbo.order_header SET updated_at = @updated_at, shipment_uuid = @shipment_uuid WHERE id = @order_id",
            parameter_values=details, enable_exception=True
        )

The consumer reads from where it last stopped, applies what it gets, and then counts what is in the
target. `first=60` is the stopping rule, and a log needs one - it has no end of its own.

One thing to know before running it. This consumer was created with `from_beginning=True`, and the
shop has been running for a while, so it starts at the **beginning** of the topic rather than at its
end. There is a backlog of thousands of messages, and it will work through them 60 at a time.

In [8]:
events = read_kfk_topic(connection=consumer, topic=TOPIC, first=60, as_type="dict")

for event in events:
    apply_event(event)

print(f"applied {len(events)} events")

invoke_sql_query(
    connection=sql_connection,
    query="SELECT (SELECT COUNT(*) FROM dbo.customer) AS customers, (SELECT COUNT(*) FROM dbo.order_header) AS orders, (SELECT COUNT(*) FROM dbo.order_detail) AS details"
)

[VERBOSE] Subscribing to topic photoservice.events
[VERBOSE] Reading, stopping after 60 messages
[VERBOSE] Retrieved 60 messages
[VERBOSE] Importing data into [dbo].[order_header]
[VERBOSE] Creating cursor
[VERBOSE] Inserting 1 rows
[VERBOSE] 1/1 rows inserted (100.0%) - 68 rows/sec
[VERBOSE] Bulk insert complete
[VERBOSE] Importing data into [dbo].[order_detail]
[VERBOSE] Creating cursor
[VERBOSE] Inserting 18 rows
[VERBOSE] 18/18 rows inserted (100.0%) - 990 rows/sec
[VERBOSE] Bulk insert complete
[VERBOSE] Importing data into [dbo].[order_header]
[VERBOSE] Creating cursor
[VERBOSE] Inserting 1 rows
[VERBOSE] 1/1 rows inserted (100.0%) - 57 rows/sec
[VERBOSE] Bulk insert complete
[VERBOSE] Importing data into [dbo].[order_detail]
[VERBOSE] Creating cursor
[VERBOSE] Inserting 3 rows
[VERBOSE] 3/3 rows inserted (100.0%) - 181 rows/sec
[VERBOSE] Bulk insert complete
[VERBOSE] Importing data into [dbo].[order_header]
[VERBOSE] Creating cursor
[VERBOSE] Inserting 1 rows
[VERBOSE] 1/1 rows

,customers,orders,details
0,6,57,728


### Now run that cell again

`applied 60 events` again, and the totals climb. Two numbers doing two different jobs, and they are
worth separating:

* **`applied ...`** is the increment - what this one call did. It says exactly 60 because 60 is the
  cap and there is still a backlog behind it. Once the consumer catches up with the end of the topic
  it will drop to a handful: whatever the shop managed while you were reading this.
* **The counts** are totals in the target. They only ever go up, because that is what a target does.

That is the offset at work. The consumer committed its position as it read, so each call starts where
the last one stopped. It is *catching up*, one bite at a time, and nothing is re-read or skipped.

How far behind it is has a name: **lag**, the distance between a consumer position and the end of the
topic. Redpanda Console shows it per group, and it is the number anybody running Kafka in earnest
watches.

Compare all of that with demo 4, where every transfer began by asking SQL Server for `MAX(id)`. Here
nobody asked the target anything - the consumer already knew.

**And note what the group id is not doing.** Each run of this notebook invents new group names, so a
restarted kernel begins from the start of the topic again - but *within* one session the consumer and
its group stay put, which is exactly why re-running the cell moves forward instead of repeating.

### And now the part a database comparison cannot do

The events are still on the topic. They were not consumed away - reading is not taking.

So a reader that has never read before can have the whole history, in order, and build the target
from nothing. `group_id` is what Kafka remembers a reader by, so a name nobody has used is a reader
with no past.

There is one thing to settle first, and it is easy to miss: **a replay of a live topic has no
natural end either.** The shop is still producing, so "read everything" would mean "read forever" -
`read_kfk_topic` would keep finding new messages and never see its five seconds of quiet. The **high
watermark** is where "everything" stops: the offset one past the last message the broker holds right
now. Catching up and keeping up are the same operation, started in different places.

In [9]:
from confluent_kafka import TopicPartition

# The tables have rows in them from the run above, and replaying an insert onto a row that is
# already there is a primary key violation. Replay is only worth anything if applying an event
# twice is safe - here that is bought the blunt way, by starting from empty.
for query in ["TRUNCATE TABLE dbo.customer", "TRUNCATE TABLE dbo.order_header", "TRUNCATE TABLE dbo.order_detail"]:
    invoke_sql_query(connection=sql_connection, query=query, enable_exception=True)

replay = connect_kfk_consumer(
    instance="127.0.0.1:19092",
    group_id=f"replay-{RUN}",
    from_beginning=True
)

# Where does "the whole topic" end? The shop is still producing, so the answer is a moving
# target - and this is the high watermark, the offset one past the last message the broker has
# right now. Without it, "read everything" means "read forever".
low, high = replay.get_watermark_offsets(TopicPartition(TOPIC, 0), timeout=10)
print(f"the topic holds {high - low} messages")

events = read_kfk_topic(connection=replay, topic=TOPIC, first=high - low, as_type="dict")

print(f"read {len(events)} of them")

[VERBOSE] Creating cursor
[VERBOSE] Executing query
[VERBOSE] Non-query executed, rowcount=-1
[VERBOSE] Creating cursor
[VERBOSE] Executing query
[VERBOSE] Non-query executed, rowcount=-1
[VERBOSE] Creating cursor
[VERBOSE] Executing query
[VERBOSE] Non-query executed, rowcount=-1
[VERBOSE] Creating consumer for instance [127.0.0.1:19092] in group [replay-112810]
[VERBOSE] Checking the connection
[VERBOSE] Returning consumer
the topic holds 18121 messages
[VERBOSE] Subscribing to topic photoservice.events
[VERBOSE] Reading, stopping after 18121 messages
[VERBOSE] Retrieved 18121 messages
read 18121 of them


In [10]:
# Applying the history one event at a time would be one round trip per event, and there are
# thousands. But the whole history is in hand, so it can be *folded* first: a payment and a
# shipment only set columns on an order that is already here, so they are applied to the row in
# memory rather than as an UPDATE against the database. What is left is three bulk inserts.
#
# Having the history is exactly what makes that possible. The incremental transfer above could
# not do it - it only ever saw the next sixty events.
customers = []
headers = {}
lines = []

for event in events:
    body = event["Details"]

    if event["Message"] == "Added customer":
        customers.append(body)

    elif event["Message"] == "Added order header":
        headers[body["id"]] = body

    elif event["Message"] == "Added order details":
        # The only event whose Details is a list - one order, many lines
        lines.extend(body)

    elif event["Message"] in ("Added payment", "Added shipment"):
        header = headers.get(body["order_id"])
        if header:
            header["updated_at"] = body["updated_at"]
            header["payment_uuid"] = body.get("payment_uuid", header["payment_uuid"])
            header["shipment_uuid"] = body.get("shipment_uuid", header["shipment_uuid"])

for table, rows in [
    ("dbo.customer", customers),
    ("dbo.order_header", list(headers.values())),
    ("dbo.order_detail", lines)
]:
    if rows:
        write_sql_table(connection=sql_connection, table=table, data=pd.DataFrame(rows), enable_exception=True)

invoke_sql_query(
    connection=sql_connection,
    query="SELECT (SELECT COUNT(*) FROM dbo.customer) AS customers, (SELECT COUNT(*) FROM dbo.order_header) AS orders, (SELECT COUNT(*) FROM dbo.order_detail) AS details"
)

[VERBOSE] Importing data into [dbo].[customer]
[VERBOSE] Creating cursor
[VERBOSE] Inserting 98 rows
[VERBOSE] 98/98 rows inserted (100.0%) - 1462 rows/sec
[VERBOSE] Bulk insert complete
[VERBOSE] Importing data into [dbo].[order_header]
[VERBOSE] Creating cursor
[VERBOSE] Inserting 4716 rows
[VERBOSE] 1000/4716 rows inserted (21.2%) - 6962 rows/sec
[VERBOSE] 2000/4716 rows inserted (42.4%) - 7629 rows/sec
[VERBOSE] 3000/4716 rows inserted (63.6%) - 8034 rows/sec
[VERBOSE] 4000/4716 rows inserted (84.8%) - 8451 rows/sec
[VERBOSE] 4716/4716 rows inserted (100.0%) - 8324 rows/sec
[VERBOSE] Bulk insert complete
[VERBOSE] Importing data into [dbo].[order_detail]
[VERBOSE] Creating cursor
[VERBOSE] Inserting 67399 rows
[VERBOSE] 1000/67399 rows inserted (1.5%) - 1801 rows/sec
[VERBOSE] 2000/67399 rows inserted (3.0%) - 2738 rows/sec
[VERBOSE] 3000/67399 rows inserted (4.5%) - 3350 rows/sec
[VERBOSE] 4000/67399 rows inserted (5.9%) - 3892 rows/sec
[VERBOSE] 5000/67399 rows inserted (7.4%) - 

,customers,orders,details
0,98,4716,67399


The target was rebuilt from the log alone. Nobody queried PostgreSQL to do it - the only thing this
notebook ever asked PostgreSQL was what `order_event` looked like, right at the start.

That is the argument for streaming, and it is not "it is faster". It is that the history is a thing
that exists, so a new reader can be added later and still catch up, and a target that gets corrupted
can be thrown away and rebuilt.

Two honest caveats:

* **Order is per partition.** One partition here, so the header arrives before its details. Spread
  the topic over several and that guarantee is per key, not per topic - which is why an order id is
  the obvious key.
* **`from_beginning` is not a rewind.** It sets `auto.offset.reset`, which only applies to a group
  that has no committed offset. Passing it to a group that has read before changes nothing. This is
  the single most confusing thing about Kafka for anybody new to it, and the cell above works
  because the group name is new every time.

### Looking at the topic

Redpanda Console is at **http://127.0.0.1:8080** - the topic, the messages, and the consumer groups
with their current offsets, including the throwaway ones this notebook has been creating.

It is there for the same reason pgAdmin is: during a talk it is easier to point at.

### Key takeaways

* Three of the four ways to know what changed compare the target against the source. The fourth is
  the application saying what it did, and it is the only one that does not have to guess.
* If you already write an outbox table, you already have events. Kafka is what that becomes once
  you want more than one reader and stop wanting to delete rows.
* Reading is not taking. The messages stay, each reader keeps its own position, and a new reader
  gets the whole history.
* Replay is only useful if applying an event twice is safe. Deciding that is the work; the
  transport is not.
* Holding the whole history lets you fold it before you write. Thousands of updates collapsed
  into three bulk inserts, because a payment only changes a row that was already in hand.
* JSON has no date and no decimal type, so the sender decides how they travel. That is the same
  lesson as `Decimal128` in MongoDB and the milliseconds in Oracle.

### Cleanup

In [11]:
for query in ["DROP TABLE dbo.customer", "DROP TABLE dbo.order_header", "DROP TABLE dbo.order_detail"]:
    invoke_sql_query(connection=sql_connection, query=query, enable_exception=True)

consumer.close()
replay.close()
sql_connection.close()
pg_connection.close()

[VERBOSE] Creating cursor
[VERBOSE] Executing query
[VERBOSE] Non-query executed, rowcount=-1
[VERBOSE] Creating cursor
[VERBOSE] Executing query
[VERBOSE] Non-query executed, rowcount=-1
[VERBOSE] Creating cursor
[VERBOSE] Executing query
[VERBOSE] Non-query executed, rowcount=-1
